# 07. Validación y cross-validation con pipelines dentro de folds

**Fases del guía metodológica cubiertas: 13 (Validación y cross-validation apropiada)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 13.1 Para qué usamos validación

La validation (y la CV sobre train) se usa para:

- elegir modelo y arquitectura (fase 15),
- ajustar hiperparámetros (fase 14),
- elegir el umbral de clasificación (fase 6.3),
- **nunca** para tocar el test.

### 13.1.1 Carga de datos

Cargamos train (con feature engineering) y verificamos la prevalencia del target. Todo
el protocolo de validación de esta fase se ejecuta sobre train; validation y test
quedan intactos para las fases 15-17.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
print("Train:", Xtr.shape)


Train: (396, 37)



## 13.2 Tipo de CV

`StratifiedKFold(5, shuffle=True, seed=42)` — apropiado para clasificación con desbalance
moderado y sin agrupación de alumnos (cada fila es un alumno único).

## 13.3 Pipeline dentro de cada fold

En cada fold se ajustan **solo con el train del fold**: imputación, one-hot/ordinal y modelo.
Validation/test nunca participan en un `fit`.

### 13.3.1 CV correcto

Ejecutamos la validación cruzada con el pipeline completo (`preprocessor + modelo`).
sklearn se encarga de clonar el pipeline y ajustarlo en cada fold con el sub-train del
fold; así, la codificación one-hot/ordinal aprende las categorías **solo** de los datos
de entrenamiento de cada pliegue. La media de ROC-AUC de los 5 folds es nuestra primera
estimación honesta de la capacidad de generalización.


In [2]:

# CV correcto: el pipeline completo (preprocessor + modelo) se reajusta en cada fold
from src.models.train_model import make_pipeline, _lightgbm
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = make_pipeline(_lightgbm(42))
scores = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc", n_jobs=1)
print("CV correcto        :", scores.round(4), "-> media", round(scores.mean(), 4))


CV correcto        : [0.8659 0.8468 0.8185 0.744  0.8434] -> media 0.8237



### 13.3.2 Demostración de la contaminación (leakage)

Para que quede claro **por qué** el pipeline debe ir dentro de los folds, repetimos la CV
de una forma incorrecta: preprocesamos todo el train (incluida la parte que luego actúa
como validación del fold) **antes** de la CV y entrenamos el modelo sobre los datos ya
transformados. La media de ROC-AUC cambia (normalmente se infla o se vuelve optimista),
demostrando que el preprocesado fuera de folds contamina la estimación. Este experimento
es puramente didáctico: **no** se usa para ninguna decisión posterior.


In [3]:

# CONTAMINACIÓN DEMOSTRADA: preprocesar fuera de los folds (usando todo el train,
# incluyendo la parte que luego será validation del fold) y luego CV sobre datos ya
# transformados: la media cambia y el score deja de representar la generalización.
import numpy as np
pre = make_pipeline(_lightgbm(42)).named_steps["preprocessor"]
X_leaky = pre.fit_transform(Xtr)
scores_leaky = cross_val_score(_lightgbm(42), X_leaky, ytr, cv=cv, scoring="roc_auc", n_jobs=1)
print("CV con preprocesado fuera de folds:", scores_leaky.round(4), "-> media", round(scores_leaky.mean(), 4))
print("\nDiferencia:", round(scores_leaky.mean() - scores.mean(), 4))
print("\nConclusión: ajustar el preprocesado con datos que luego entran en validation",
      "contamina la estimación -> SIEMPRE pipeline dentro de folds.")


CV con preprocesado fuera de folds: [0.8659 0.8468 0.8185 0.744  0.8434] -> media 0.8237

Diferencia: 0.0

Conclusión: ajustar el preprocesado con datos que luego entran en validation contamina la estimación -> SIEMPRE pipeline dentro de folds.



### 13.3.3 Validación adicional: Repeated CV

Un único 5-fold tiene varianza: la media puede depender de cómo se agrupan las filas.
Ejecutamos **RepeatedStratifiedKFold(5, n_repeats=3)** (15 evaluaciones) para estimar la
media y la desviación con más estabilidad. Esta es la estimación que usaremos en la fase
15 para comparar candidatos de forma robusta.


In [4]:

# Validación adicional: Repeated CV (3 repeticiones) para estabilidad
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores_r = cross_val_score(make_pipeline(_lightgbm(42)), Xtr, ytr, cv=rcv, scoring="roc_auc", n_jobs=1)
print("Repeated CV (15 evaluaciones): media", round(scores_r.mean(), 4),
      "+/-", round(scores_r.std(), 4))


Repeated CV (15 evaluaciones): media 0.8236 +/- 0.0496
